In [6]:
import pandas as pd
import math

df = pd.read_csv("../anglavy99_output.csv")

In [47]:
header_text = rf"""\begin{{sidewaystable}}
\centering
\addtolength{{\tabcolsep}}{{-2pt}} 
\small
\begin{{threeparttable}}
\caption{{Class size effects on test scores}} 
\vspace{{-0.5em}}
\centering 
\begin{{tabular}}{{c c c c c c c c c c c c c}} 
\hline"""

def subheader_text(test):

    test_label = "(a) Verb" if test == "verb" else "(b) Mathematics"

    text = rf"""\multicolumn{{13}}{{c}}{{{test_label} test scores}} \\
\hline
\hline  
$h$ & $n_h$ & $\hat{{\tau}}_{{2SLS}}$ & $\hat{{\tau}}_{{1}}$ & $\hat{{\tau}}_{{1}}^{{BC}}$ & $\hat{{\tau}}_{{\Lambda(1)}}$ & $\hat{{\tau}}_{{\Lambda(4)}}$ & $\mcC_{{2SLS}}$ & $STD$ & $RBC$ & $\mcC_{{\Lambda(1)}}$ & $\mcC_{{\Lambda(4)}}$ & $AR_2$ \T \\"""
    
    return text

In [65]:
def format_data_point(
    num: float,
    column_idx: int,
) -> str:
    """Format scalar values for LaTeX tables."""

    if column_idx in (2, 3):
        return f"{num:.0f}"

    if math.isinf(num):
        return r"\infty" if num > 0 else r"-\infty"

    return f"{num:.2f}"


def format_confidence_interval(
    ci_lower: float,
    ci_upper: float,
) -> str:
    """Generate confidence interval string"""

    if math.isinf(ci_lower) and math.isinf(ci_upper):
        return r"$(-\infty, \infty)$"
    if math.isinf(ci_lower):
        return rf"$(-\infty, {ci_upper:.2f}]$"
    if math.isinf(ci_upper):
        return rf"$[{ci_lower:.2f}, \infty)$"
    return rf"[{ci_lower:.2f}, {ci_upper:.2f}]"

In [70]:

def row_string(
        df: pd.DataFrame, 
        row: int
    ) -> str:
    """Generate row of data as string"""

    row_info = df.iloc[row]

    bandwidth = f"{row_info.iloc[2]}"
    sample_size = f"{row_info.iloc[3]}"

    dgp_info_and_estimates =  [format_data_point(row_info.iloc[column_idx], column_idx) for column_idx in range(6, 11)]

    confidence_intervals = [format_confidence_interval(row_info.iloc[column_idx], row_info.iloc[column_idx+1]) for column_idx in range(11, 23, 2)]

    data_points = dgp_info_and_estimates + confidence_intervals

    data_string = " & ".join(data_points)
    full_row = bandwidth + " & " + sample_size + " & " + data_string + r" \\"
    
    return full_row

def generate_panel(
        df: pd.DataFrame, 
        test: str = "verb",
    ) -> str:
    """Create panel of table"""

    rows_per_cutoff = (len(df) // 2)

    start_row = 0 if test == "verb" else rows_per_cutoff
    end_row = start_row + rows_per_cutoff       

    panel_list = [r"\hline"]

    for row_idx in range(start_row, end_row):
        panel_list.append(row_string(df, row_idx))

    panel_list.append(r'\hline')

    return "\n".join(panel_list)

def extract_bandwidth_values(
        df: pd.DataFrame,
        test: str
    ) -> dict:
    """Extract bandwidth values for each cutoff from the dataframe"""
    
    cutoffs = [40]
    bandwidth_values = {}
    
    # Filter dataframe for the specific test
    test_df = df[df['test'] == test]
    
    for cutoff in cutoffs:
        cutoff_df = test_df[test_df['cutoff'] == cutoff]
        if not cutoff_df.empty:
            # Get the first row for this cutoff (all rows have same bandwidth values)
            h_ccf = cutoff_df.iloc[0]['bw_cov']
            h_ik = cutoff_df.iloc[0]['bw_mse']
            bandwidth_values[cutoff] = {
                'h_ik': h_ik,
                'h_ccf': h_ccf
            }
    
    return bandwidth_values

footer_string = rf"""\hline
\end{{tabular}}
The MSE- and coverage optimal bandwidths are $h_{{IK}} = {df["bw_mse"][0]:.2f}$ and $h_{{CCF}} = {df["bw_cov"][0]:.2f}$ for verb test scores, and $h_{{IK}} = {df["bw_mse"][7]:.2f}$ and $h_{{CCF}} = {df["bw_cov"][7]:.2f}$ for mathematics test scores
\label{{table empirical math}}
\end{{threeparttable}}
\end{{sidewaystable}}
"""

In [71]:
table_text = []

table_text.append(header_text)
table_text.append(subheader_text("verb"))
table_text.append(generate_panel(df, "verb"))
table_text.append(subheader_text("math"))
table_text.append(generate_panel(df, "math"))
table_text.append(footer_string)

print("\n".join(table_text))

\begin{sidewaystable}
\centering
\addtolength{\tabcolsep}{-2pt} 
\small
\begin{threeparttable}
\caption{Class size effects on test scores} 
\vspace{-0.5em}
\centering 
\begin{tabular}{c c c c c c c c c c c c c} 
\hline
\multicolumn{13}{c}{(a) Verb test scores} \\
\hline
\hline  
$h$ & $n_h$ & $\hat{\tau}_{2SLS}$ & $\hat{\tau}_{1}$ & $\hat{\tau}_{1}^{BC}$ & $\hat{\tau}_{\Lambda(1)}$ & $\hat{\tau}_{\Lambda(4)}$ & $\mcC_{2SLS}$ & $STD$ & $RBC$ & $\mcC_{\Lambda(1)}$ & $\mcC_{\Lambda(4)}$ & $AR_2$ \T \\
\hline
6 & 149 & -0.12 & -0.12 & -0.23 & -0.10 & -0.07 & [-0.28, 0.05] & [-0.39, 0.14] & [-0.79, 0.32] & [-0.23, 0.03] & [-0.15, 0.01] & $(-\infty, \infty)$ \\
8 & 229 & -0.09 & -0.10 & -0.15 & -0.09 & -0.08 & [-0.17, -0.01] & [-0.23, 0.02] & [-0.38, 0.09] & [-0.16, -0.01] & [-0.14, -0.01] & [-0.78, 0.10] \\
10 & 295 & -0.06 & -0.08 & -0.13 & -0.06 & -0.06 & [-0.11, -0.01] & [-0.16, -0.00] & [-0.27, 0.00] & [-0.11, -0.01] & [-0.10, -0.01] & [-0.26, -0.02] \\
12 & 379 & -0.05 & -0.07 & -0.11 

In [40]:
extract_bandwidth_values(df, "math")


{40: {'h_ik': np.float64(10.8280355543559),
  'h_ccf': np.float64(7.39450457226129)}}

In [ ]:
footer_string = rf"""The MSE- and coverage optimal bandwidths are $h_{{IK}} = {df["bw_mse"][0]:.2f}$ and $h_{{CCF}} = {df["bw_cov"][0]:.2f}$ for verb test scores, and $h_{{IK}} = {df["bw_mse"][7]:.2f}$ and $h_{{CCF}} = {df["bw_cov"][7]:.2f}$ for mathematics test scores"""
print(footer_string)

The MSE- and coverage optimal bandwidths are $h_{IK} = 9.75$ and $h_{CCF} = 6.66$ for verb scores, and $h_{IK} = 10.83$ and $h_{CCF} = 7.39$ for mathematics scores
